# Measuring Training and Prediction Time (Basics)
Use `time.perf_counter()` for timing code: it is a high-resolution, monotonic clock.
`time.time()` is wall-clock time - lower resolution on some systems and it can jump if the system clock is adjusted.

In [1]:
import time
import numpy as np
from sklearn.linear_model import LinearRegression  # Example model

rng = np.random.default_rng(42)

# Generate some sample data (replace with your actual data)
X_train = rng.random((10000, 10))  # 10,000 samples, 10 features
y_train = rng.random(10000)
X_test = rng.random((1000, 10))    # 1,000 samples, 10 features

In [2]:
# 1. Training Time Measurement

model = LinearRegression()  # Initialize your model

start_time_train = time.perf_counter()  # Record start time
model.fit(X_train, y_train)             # Train the model
end_time_train = time.perf_counter()    # Record end time

training_time = end_time_train - start_time_train
print(f"Training time: {training_time:.4f} seconds")

Training time: 0.0120 seconds


In [3]:
# 2. Prediction Time Measurement (for a single prediction)

# Warm-up: the very first call often pays one-time costs (imports, memory allocation, caches)
model.predict(X_test[:1])

# Option 1: Measuring time for a single prediction
start_time_predict_single = time.perf_counter()
single_prediction = model.predict(X_test[0].reshape(1, -1))  # Predict for one data point
end_time_predict_single = time.perf_counter()
prediction_time_single = end_time_predict_single - start_time_predict_single
print(f"Single prediction time: {prediction_time_single * 1e6:.1f} microseconds")

Single prediction time: 246.4 microseconds


In [4]:
# Option 2: Measuring time for multiple predictions (more realistic)
num_predictions = len(X_test)  # or any number you like

start_time_predict_multiple = time.perf_counter()
predictions = model.predict(X_test)  # Predict for all (or a subset) of the test data
end_time_predict_multiple = time.perf_counter()

prediction_time_multiple = end_time_predict_multiple - start_time_predict_multiple
print(f"Prediction time for {num_predictions} samples: {prediction_time_multiple * 1e3:.3f} ms")

average_prediction_time = prediction_time_multiple / num_predictions
print(f"Average prediction time per sample: {average_prediction_time * 1e6:.3f} microseconds")
print(f"Batching speed-up vs. one-at-a-time: {prediction_time_single / average_prediction_time:.0f}x")

Prediction time for 1000 samples: 0.390 ms
Average prediction time per sample: 0.390 microseconds
Batching speed-up vs. one-at-a-time: 632x


In [5]:
# Important Considerations:

# - Data Size: The size of your training and test data significantly impacts the times.  Run with representative data.
# - Model Complexity: More complex models (e.g., deep neural networks) will take longer to train and predict.
# - Hardware:  Your CPU, RAM, and whether you're using a GPU will affect the times.
# - Averaging: Run the timing multiple times and report the mean AND spread to reduce the impact of system variations.
# - First Prediction: The *very first* call is often slower due to initialization overhead - do a warm-up first.
# - Batch Size (for prediction): Processing data in batches is usually much more efficient per sample.
# - Profiling: For more detailed analysis, use a profiler (e.g., `cProfile`) - see 10_W_MeasuringTime.ipynb.

# Example of averaging using a loop:
num_runs = 10
run_times = []
for _ in range(num_runs):
    start_time_train = time.perf_counter()
    model.fit(X_train, y_train)
    run_times.append(time.perf_counter() - start_time_train)

print(f"Training time over {num_runs} runs: {np.mean(run_times) * 1e3:.2f} ms "
      f"+/- {np.std(run_times) * 1e3:.2f} ms (min {np.min(run_times) * 1e3:.2f} ms)")

Training time over 10 runs: 2.69 ms +/- 0.10 ms (min 2.57 ms)


In [6]:
# The standard library's timeit module does the repeat loop for you
import timeit

times = timeit.repeat(lambda: model.predict(X_test), number=100, repeat=5)
per_call = np.array(times) / 100
print(f"predict(1000 rows): best {per_call.min() * 1e3:.3f} ms, median {np.median(per_call) * 1e3:.3f} ms")

# In a notebook the %timeit magic is the shortest form:
%timeit model.predict(X_test)

predict(1000 rows): best 0.086 ms, median 0.091 ms


85.6 μs ± 3.43 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
